In [40]:
import pandas as pd
import numpy as np

In [41]:
# Ruta Modificable
ruta_sucio = "C:/Users/HP/Desktop/DataPool/Proyecto_Transvial/Info_Archivos_Anteriores_BI/"
ruta_limpio = "C:/Users/HP/Desktop/DataPool/Proyecto_Transvial/Información_Proyecto_BI/"

In [42]:
# DataFrame por tipo de flota
df_12metros = pd.read_excel(
    f"{ruta_sucio}Historico_km.xlsm", 
    sheet_name="KM12", 
    engine="calamine",
    header=1,
    usecols="A:E")

df_18metros = pd.read_excel(
    f"{ruta_sucio}Historico_km.xlsm", 
    sheet_name="KM18", 
    engine="calamine",
    header=0,
    usecols="A:E")

df_9metros = pd.read_excel(
    f"{ruta_sucio}Historico_km.xlsm", 
    sheet_name="KM09", 
    engine="calamine",
    header=0,
    usecols="A:E")


In [43]:
df_km_fin = pd.concat([df_12metros, df_18metros, df_9metros], ignore_index = True)
df_km_fin = df_km_fin[(df_km_fin["FECHA"] > "2023-12-31") & (df_km_fin["FECHA"] <= "2025-05-05")]
df_km_fin = df_km_fin.rename(columns={
    "FECHA" : "Fecha_Km",
    "Nº INTERNO" : "Cod_Bus",
    "KM INICIAL" : "Km_Ini",
    "KM FINAL" : "Km_Fin",
    "DIFERENCIA KM" : "Km_Diario"
})
df_km_fin = df_km_fin.drop_duplicates(subset = ["Fecha_Km", "Cod_Bus"], keep = "first")

df_km_fin["Fecha_Km"] = pd.to_datetime(df_km_fin["Fecha_Km"], errors="coerce")

str_cols = df_km_fin.select_dtypes(include=["str", "string"]).columns
df_km_fin[str_cols] = df_km_fin[str_cols].astype(object)

df_km_fin.to_parquet(
    f"{ruta_limpio}Fact_Km_Historico.parquet",
    index = False,
    engine = "pyarrow",
    coerce_timestamps = "ms",
    allow_truncated_timestamps = True
)